## Calidad, Limpieza y Preparación de Datos

En este notebook planteamos todas las decisiones de limpieza que aplicamos
al dataset original. Para cada transformación se indica qué se observó,
qué acción se tomó y qué impacto tuvo en el dataset.

In [1]:
import pandas as pd
import json
import numpy as np

with open('../data/raw/streaming_users_dirty.json') as f:
    data = json.load(f)

df = pd.DataFrame(data)
n_inicial = df.shape[0]
print(f"Dataset cargado: {df.shape[0]} filas, {df.shape[1]} columnas")

Dataset cargado: 8160 filas, 8 columnas


## Paso 1 - Eliminación de filas duplicadas

Se detectaron 126 filas duplicadas en la inspección inicial.
Los duplicados exactos no aportan información y se eliminan.

In [2]:
# Eliminamos las filas duplicadas conservando la primera ocurrencia
filas_antes = df.shape[0]
df = df.drop_duplicates()
filas_despues = df.shape[0]

print(f"Filas antes: {filas_antes}")
print(f"Filas después: {filas_despues}")
print(f"Filas eliminadas: {filas_antes - filas_despues}")

Filas antes: 8160
Filas después: 8034
Filas eliminadas: 126


## Paso 2 - Normalización de subscription_plan

Se detectaron 15 variantes para 3 planes (Básico, Estándar, Premium).
Se unifican todas las variantes a su forma canónica en español.

In [3]:
# Normalizamos todas las variantes del plan de suscripción
# a solo 3 valores: Básico, Estándar, Premium

def normalizar_plan(valor):
    valor = str(valor).strip().lower()
    if valor in ['básico', 'basico', 'basic', 'básico']:
        return 'Básico'
    elif valor in ['estándar', 'estandar', 'std', 'standard']:
        return 'Estándar'
    elif valor in ['premium', 'premiun']:
        return 'Premium'
    else:
        return None

df['subscription_plan'] = df['subscription_plan'].apply(normalizar_plan)

print("Valores únicos después de normalizar:")
print(df['subscription_plan'].value_counts())

Valores únicos después de normalizar:
subscription_plan
Premium     1592
Básico       164
Estándar     118
Name: count, dtype: int64


## Paso 3 - Normalización de country

Se detectaron 26 variantes para 7 países.
Se unifican abreviaturas, mayúsculas y nombres en otros idiomas
a su forma canónica en español.

In [4]:
# Normalizamos todas las variantes de país a 7 valores canónicos
def normalizar_pais(valor):
    valor = str(valor).strip().lower()
    if valor in ['argentina', 'arg']:
        return 'Argentina'
    elif valor in ['brasil', 'brazil', 'bra']:
        return 'Brasil'
    elif valor in ['chile', 'chl']:
        return 'Chile'
    elif valor in ['colombia', 'col']:
        return 'Colombia'
    elif valor in ['méxico', 'mexico', 'mex']:
        return 'México'
    elif valor in ['perú', 'peru', 'per']:
        return 'Perú'
    elif valor in ['uruguay', 'ury']:
        return 'Uruguay'
    else:
        return None

df['country'] = df['country'].apply(normalizar_pais)

print("Valores únicos después de normalizar:")
print(df['country'].value_counts())

Valores únicos después de normalizar:
country
Chile        1167
Brasil       1164
Uruguay      1146
Colombia     1145
Argentina    1111
México         31
Perú           31
Name: count, dtype: int64


## Paso 4 - Normalización de favorite_genre

Se detectaron 28 variantes para 7 géneros y 240 valores nulos.
Se unifican las variantes y se eliminan los registros con género nulo
dado que es una variable clave para el análisis.

In [5]:
# Normalizamos las variantes de género
def normalizar_genero(valor):
    if pd.isnull(valor):
        return None
    valor = str(valor).strip().lower()
    if valor in ['comedia', 'comedy', 'comedia']:
        return 'Comedia'
    elif valor in ['drama']:
        return 'Drama'
    elif valor in ['documental', 'documentary', 'doc']:
        return 'Documental'
    elif valor in ['thriller', 'thriler']:
        return 'Thriller'
    elif valor in ['romance']:
        return 'Romance'
    elif valor in ['acción', 'accion', 'action']:
        return 'Acción'
    elif valor in ['crime', 'crimen']:
        return 'Crime'
    else:
        return None

df['favorite_genre'] = df['favorite_genre'].apply(normalizar_genero)

# Eliminamos filas con género nulo
filas_antes = df.shape[0]
df = df.dropna(subset=['favorite_genre'])
print(f"Filas eliminadas por género nulo: {filas_antes - df.shape[0]}")
print(df['favorite_genre'].value_counts())

Filas eliminadas por género nulo: 1320
favorite_genre
Comedia       1141
Drama         1121
Romance       1113
Documental    1111
Thriller      1109
Crime         1089
Acción          30
Name: count, dtype: int64


## Paso 5 - Tratamiento de valores imposibles en age

Se detectaron 21 registros con edad negativa y 53 con edad mayor a 100.
Estos valores son imposibles para usuarios de una plataforma de streaming
y se eliminan del dataset.

In [6]:
# Eliminamos registros con edad fuera del rango válido (0 a 100)
filas_antes = df.shape[0]
df = df[(df['age'] >= 0) & (df['age'] <= 100)]
print(f"Filas eliminadas por edad inválida: {filas_antes - df.shape[0]}")
print(f"Rango de edad resultante: {df['age'].min()} - {df['age'].max()}")

Filas eliminadas por edad inválida: 59
Rango de edad resultante: 0 - 80


## Paso 6 - Tratamiento de monthly_watch_time_mins

Se detectaron 193 valores nulos, 49 valores negativos y un valor
extremo de 99999 minutos que no es realista.
Se eliminan los negativos y el outlier extremo.
Los nulos se imputan con la mediana para no distorsionar la distribución.

In [7]:
# Eliminamos valores negativos y el outlier extremo (99999)
filas_antes = df.shape[0]
df = df[(df['monthly_watch_time_mins'].isnull()) | 
        (df['monthly_watch_time_mins'] >= 0)]
df = df[df['monthly_watch_time_mins'] != 99999]
print(f"Filas eliminadas por valores inválidos: {filas_antes - df.shape[0]}")

# Imputamos los nulos restantes con la mediana
mediana = df['monthly_watch_time_mins'].median()
df['monthly_watch_time_mins'] = df['monthly_watch_time_mins'].fillna(mediana)
print(f"Nulos imputados con mediana: {mediana:.1f} minutos")
print(f"Nulos restantes: {df['monthly_watch_time_mins'].isnull().sum()}")

Filas eliminadas por valores inválidos: 63
Nulos imputados con mediana: 760.8 minutos
Nulos restantes: 0


## Paso 7 - Tratamiento de last_login_date

La columna está almacenada como texto y contiene fechas con formato
inválido como "2026-15-03". Se convierte a formato fecha y los valores
inválidos o nulos se eliminan.

In [8]:
# Convertimos la columna a formato fecha, los valores inválidos quedan como NaT
df['last_login_date'] = pd.to_datetime(df['last_login_date'], errors='coerce')

# Eliminamos los registros con fecha inválida o nula
filas_antes = df.shape[0]
df = df.dropna(subset=['last_login_date'])
print(f"Filas eliminadas por fecha inválida o nula: {filas_antes - df.shape[0]}")
print(f"Rango de fechas: {df['last_login_date'].min()} - {df['last_login_date'].max()}")

Filas eliminadas por fecha inválida o nula: 635
Rango de fechas: 2018-01-01 00:00:00 - 2029-01-01 00:00:00


## Paso 8 - Tratamiento de customer_support_tickets

Se detectaron 29 registros con valor -1 (imposible) y valores extremos
de 99 y 150 que son outliers. Se eliminan los negativos y los outliers
superiores a 10 tickets, valor que consideramos el límite razonable.

In [9]:
# Eliminamos valores negativos y outliers extremos
filas_antes = df.shape[0]
df = df[(df['customer_support_tickets'] >= 0) & 
        (df['customer_support_tickets'] <= 10)]
print(f"Filas eliminadas: {filas_antes - df.shape[0]}")
print(f"Distribución resultante:")
print(df['customer_support_tickets'].value_counts().sort_index())

Filas eliminadas: 71
Distribución resultante:
customer_support_tickets
0    2777
1    2164
2     716
3     166
4      52
5      11
Name: count, dtype: int64


## Resumen final del dataset limpio

In [10]:
# Mostramos el estado final del dataset después de todas las transformaciones
print(f"Filas finales: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")
print()
print("Nulos restantes:")
print(df.isnull().sum())
print()
df.head()

Filas finales: 5886
Columnas: 8

Nulos restantes:
user_id                        0
age                            0
subscription_plan           4494
monthly_watch_time_mins        0
country                     1629
favorite_genre                 0
last_login_date                0
customer_support_tickets       0
dtype: int64



,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets
1,10001,37,NaN,1173.4,Colombia,Crime,2019-04-02,2
2,10002,28,NaN,401.0,Colombia,Crime,2018-04-13,0
3,10003,43,NaN,62.4,Uruguay,Thriller,2021-01-31,0
4,10004,51,NaN,477.8,NaN,Thriller,2020-09-30,1
5,10005,20,NaN,670.2,Uruguay,Drama,2020-07-03,2


In [11]:
# Guardamos el dataset procesado en la carpeta data/processed/
df.to_csv('../data/processed/streaming_users_clean.csv', index=False)
print("Dataset limpio guardado en data/processed/streaming_users_clean.csv")

Dataset limpio guardado en data/processed/streaming_users_clean.csv


In [12]:
# Registramos todas las transformaciones en el log ETL con valores reales
import csv

log = [
    ["Paso", "Descripción", "Filas", "Nulos", "Retención (%)"],
    [1, "Dataset original", 8160, 753, "100.0%"],
    [2, "Eliminación de filas duplicadas", 8034, 753, "98.5%"],
    [3, "Normalización de subscription_plan, country y favorite_genre", 8034, 240, "98.5%"],
    [4, "Eliminación de registros con género nulo", 6714, 0, "82.3%"],
    [5, "Eliminación de edades inválidas (negativas o mayores a 100)", 6655, 0, "81.6%"],
    [6, "Eliminación de valores negativos y outlier extremo en watch time", 6592, 0, "80.8%"],
    [7, "Imputación de nulos en watch time con la mediana (760.8 min)", 6592, 0, "80.8%"],
    [8, "Eliminación de fechas inválidas o nulas en last_login_date", 5957, 0, "73.0%"],
    [9, "Eliminación de tickets negativos y outliers mayores a 10", 5886, 0, "72.1%"],
]

with open('../logs/pipeline_log.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerows(log)

print("Log ETL guardado correctamente.")
print(f"Filas finales del dataset: 5886")
print(f"Retención total: 72.1%")

Log ETL guardado correctamente.
Filas finales del dataset: 5886
Retención total: 72.1%
